# The Oregon Medicaid experiment

This notebook reproduces the results from Chapter 4.1 of the Approximate Maximum Perturbation (AMIP) paper using Python and the JAX library.

In [102]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import os
import pandas as pd
import numpy as np

# 1. Study and Data Preparation

## 1.1 Data Requirements

To reproduce the findings, I downloaded the Oregon Medicaid survey responder dataset from the [original link](https://www.nber.org/research/data/oregon-health-insurance-experiment-data). The data preparation script was retained from the AMIP paper's [Github Repo](https://github.com/rgiordan/AMIPPaper) and is saved in `data/data_processing/prepare_OHIE_data.R`

In [103]:
# Load the CSV file into memory
df = pd.read_csv("/Users/yumin/Documents/Research/PML/Repository/amip-robustness-jax/data/OHIE_Public_Use_Files/OHIE_Data/data_for_analysis_R.csv")

print(f"The shape of the dataset before filtering: {df.shape}")

The shape of the dataset before filtering: (74922, 563)


/var/folders/mr/gyhq4nd5159cfdvmmd1l_d440000gn/T/ipykernel_77868/3469417847.py:2: DtypeWarning: Columns (0: female_cb) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/yumin/Documents/Research/PML/Repository/amip-robustness-jax/data/OHIE_Public_Use_Files/OHIE_Data/data_for_analysis_R.csv")


In [104]:
def prepare_ohie_regression_data(orig_df, outcome_var):
    """
    Prepares the dataset for OHIE regression modeling for a specific outcome.
    """
    # 1. Filter for the 12-month survey respondents
    df = orig_df[orig_df['sample_12m_resp'] == 1].copy()
    
    # 2. Identify fixed effect columns (starting with "ddd")
    ddd_cols = [col for col in df.columns if col.startswith('ddd')]
    
    # 3. Define the core variables used in the model
    core_vars = [
        'treatment',             # Treatment indicator
        'household_id',          # Cluster ID for standard errors
        outcome_var,             # Target outcome variable
        'ohp_all_ever_survey',   # Endogenous instrument variable (for IV/LATE)
        'weight_12m'             # Final 12-month survey weights
    ]
    
    # 4. Subset the dataframe to only the variables needed for this specific model
    cols_to_keep = core_vars + ddd_cols
    regression_df = df[cols_to_keep].copy()
    
    # 5. Filter out missing values (equivalent to R's complete.cases)
    regression_df_clean = regression_df.dropna()
    
    # 6. Re-index the household_id cluster index (as performed in the R script)
    regression_df_clean.loc[:, 'household_id'] = (
        regression_df_clean['household_id'] - regression_df_clean['household_id'].min()
    )
    
    # Extract the prepared components for modeling
    y = regression_df_clean[outcome_var]
    X_treatment = regression_df_clean['treatment']
    X_instrument = regression_df_clean['ohp_all_ever_survey']
    X_fixed_effects = regression_df_clean[ddd_cols]
    weights = regression_df_clean['weight_12m']
    clusters = regression_df_clean['household_id']
    
    return regression_df_clean, y, X_treatment, X_instrument, X_fixed_effects, weights, clusters

In [105]:
table_9_outcomes = [
    "health_genflip_bin_12m", 
    "health_notpoor_12m",
    "health_chgflip_bin_12m", 
    "notbaddays_tot_12m",
    "notbaddays_phys_12m", 
    "notbaddays_ment_12m",
    "nodep_screen_12m"
]

## 1.2 The Models

The AMIP paper evaluates both Intent-to-Treat (ITT) and Local Average Treatment Effect (LATE) models.

* **ITT (OLS):** $y_{ihj} = \beta_0 + \beta_1 \text{LOTTERY}_h + \beta_2 X_{ih} + \beta_3 V_{ih} + \epsilon_{ihj}$.


* **LATE (IV):** Uses a two-stage least squares (2SLS)

    * $y_{ihj} = \pi_0 + \pi_1 \text{INSURANCE}_{ih} + \pi_2 X_{ih} + \pi_3 V_{ih} + \upsilon_{ihj}$
    * $\text{INSURANCE}_{ih} = \delta_0 + \delta_1 \text{LOTTERY}_h + \delta_2 X_{ih} + \delta_3 V_{ih} + \mu_{ihj}$

In [106]:
@jax.jit
def fit_wls(X, y, weights):
    """
    Helper function to fit a Weighted Least Squares (WLS) model.
    Multiples X and y by the square root of the weights.
    """
    # Ensure weights are a column vector for broadcasting
    sqrt_w = jnp.sqrt(weights).reshape(-1, 1)

    X_weighted = X * sqrt_w

    y_col = y.reshape(-1, 1) if y.ndim == 1 else y
    y_weighted = y_col * sqrt_w

    beta, _, _, _ = jnp.linalg.lstsq(X_weighted, y_weighted)
    return beta

@jax.jit
def itt_model_weighted(y, lottery, X, V, weights):
    """
    Weighted Intent-to-Treat (ITT) Model using OLS.
    y = beta_0 + beta_1*LOTTERY + beta_2*X + beta_3*V + error
    """
    # 1. Create the intercept column (beta_0)
    intercept = jnp.ones((y.shape[0], 1))

    # 2. Build the design matrix: [Intercept, LOTTERY, X, V]
    # Ensure all 1D arrays are reshaped to 2D column vectors if necessary
    lottery_col = lottery.reshape(-1, 1) if lottery.ndim == 1 else lottery

    Z = jnp.column_stack([intercept, lottery_col, X, V])

    beta = fit_wls(Z, y, weights)

    return beta

@jax.jit
def late_iv_model_weighted(y, insurance, lottery, X, V, weights):
    """
    Weighted Local Average Treatment Effect (LATE) Model using 2SLS.
    Stage 1: INSURANCE = delta_0 + delta_1*LOTTERY + delta_2*X + delta_3*V
    Stage 2: y = pi_0 + pi_1*INSURANCE_HAT + pi_2*X + pi_3*V
    """
    # 1. Create the intercept column
    intercept = jnp.ones((y.shape[0],1))
    lottery_col = lottery.reshape(-1, 1) if lottery.ndim == 1 else lottery
    insurance_col = insurance.reshape(-1, 1) if insurance.ndim == 1 else insurance

    # Stage 1
    Z_stage1 = jnp.column_stack([intercept, lottery_col, X, V])

    delta = fit_wls(Z_stage1, insurance_col, weights)

    insurance_hat = jnp.dot(Z_stage1, delta)

    # Stage 2
    Z_stage2 = jnp.column_stack([intercept, insurance_hat, X, V])

    pi = fit_wls(Z_stage2, y, weights)

    return delta, pi



In [107]:
itt_estimates_all = []
late_estimates_all = []

for outcome in table_9_outcomes:

    # 1. Prepare data for the current outcome
    clean_df, y, X_treat, X_inst, X_fe, w, clusters = prepare_ohie_regression_data(
        orig_df=df, 
        outcome_var=outcome
    )

    # 2. Convert the pandas Series and DataFrames to JAX arrays
    y_jax = jnp.array(y.values)
    lottery_jax = jnp.array(X_treat.values)
    insurance_jax = jnp.array(X_inst.values)
    X_jax = jnp.array(X_fe.values)
    weights_jax = jnp.array(w.values)

    # 3. Handle the V matrix (Optional covariates)
    # The R script for Table 9 only utilizes the 'ddd' fixed effects (which we mapped to X).
    # To satisfy the JAX function signature without affecting the math, we pass a zero-column array for V.
    V_jax = jnp.empty((y_jax.shape[0], 0))

    # 4. Execute the ITT (OLS) Model
    itt_betas = itt_model_weighted(
        y=y_jax, 
        lottery=lottery_jax, 
        X=X_jax, 
        V=V_jax,
        weights=weights_jax
    )

    # 5. Execute the LATE (2SLS) Model
    late_deltas, late_pis = late_iv_model_weighted(
        y=y_jax, 
        insurance=insurance_jax, 
        lottery=lottery_jax, 
        X=X_jax, 
        V=V_jax,
        weights=weights_jax
    )

    # 6. Extract the coefficients of interest
    # Index 0 is the intercept, Index 1 is the treatment/insurance effect
    itt_estimate = itt_betas[1]
    late_estimate = late_pis[1]

    itt_estimates_all.append(itt_estimate)
    late_estimates_all.append(late_estimate)

    print(f"Outcome: {outcome}")
    print(f"ITT Estimate (Impact of winning the lottery): {itt_estimate[0]:.5f}")
    print(f"LATE Estimate (Impact of obtaining insurance): {late_estimate[0]:.5f}")
    

Outcome: health_genflip_bin_12m
ITT Estimate (Impact of winning the lottery): 0.03869
LATE Estimate (Impact of obtaining insurance): 0.13288
Outcome: health_notpoor_12m
ITT Estimate (Impact of winning the lottery): 0.02884
LATE Estimate (Impact of obtaining insurance): 0.09903
Outcome: health_chgflip_bin_12m
ITT Estimate (Impact of winning the lottery): 0.03278
LATE Estimate (Impact of obtaining insurance): 0.11276
Outcome: notbaddays_tot_12m
ITT Estimate (Impact of winning the lottery): 0.38097
LATE Estimate (Impact of obtaining insurance): 1.31711
Outcome: notbaddays_phys_12m
ITT Estimate (Impact of winning the lottery): 0.45914
LATE Estimate (Impact of obtaining insurance): 1.58540
Outcome: notbaddays_ment_12m
ITT Estimate (Impact of winning the lottery): 0.60295
LATE Estimate (Impact of obtaining insurance): 2.08198
Outcome: nodep_screen_12m
ITT Estimate (Impact of winning the lottery): 0.02272
LATE Estimate (Impact of obtaining insurance): 0.07831
